In [ ]:
# Jinko specifics imports & initialization
# Please fold this section and do not edit it
from jinko import JinkoClient

# Connect to Jinko (see README.md for more options)
client = JinkoClient()
client.auth_check()


In [ ]:
# Cookbook specifics imports
import json
import io
import zipfile
import pandas as pd

## Select calibration of interest

In [ ]:
"""
An optional folder location to store debug files
"""

folder = "."
# Set this to True to write debugging files
write_files = True

"""
Calibration short id can be retrieved from the URL of your calibration in Jinko, pattern is `https://jinko.ai/<calibration_short_id>`
"""
calibration_short_id = "ca-hIAx-A64d"
# Choose a spefic revision. By default we return the last version
revision = 19

calibration = client.get_calibration(calibration_short_id, revision=revision)

# Uncomment the following if you want to use the latest completed or stopped version
# calibration = client.get_calibration(calibration_short_id)
# latest_version = next(
#     version
#     for version in calibration.versions().iter()
#     if version.label in {"completed", "stopped"}
# )
# calibration = client.get_calibration(calibration_short_id, revision=latest_version.revision)

print(
    f"Picked Calibration with coreItemId: {calibration.core_id}, snapshotId: {calibration.snapshot_id}"
)

status = calibration.status()
print(f"It has status: {status}")


## Debug data tables

In [ ]:
sorted_patients = calibration.results.sorted_patients(
    sort_by="optimizationWeightedScore"
)
if len(sorted_patients) > 0:
    best_patient = sorted_patients[0]
    print(f"best_patient: {best_patient}")

    augmented_data = calibration.results.augment_data_tables(
        patient_id=best_patient["patientNumber"],
        iteration=best_patient["iteration"],
    )
    archive = zipfile.ZipFile(io.BytesIO(augmented_data))
    filenames = archive.namelist()
    print(f"CSV files in archive: {filenames}")
    for filename in filenames:
        print(f"Extracting : {filename}")
        csv_content = archive.read(filename).decode("utf-8")
        datatable = pd.read_csv(io.StringIO(csv_content))
        display(datatable)


## Debug scoring or solving errors

In [ ]:
# Choose the iteration number you wish to debug, 1 is a good guess if you do not know where to start
iteration = 1

scoring_errors = calibration.results.errors(
    scalar_id="optimizationWeightedScore",
    iteration=iteration,
)
if len(scoring_errors["errors"]) <= 0:
    print("No error for this iteration")
else:
    first_error = scoring_errors["errors"][0]
    patient_id = first_error["patientId"]
    print(f"Patient {patient_id} has an error:\n{first_error['scoringError']}")


Based on the above information, you may want to inspect the solver output for a given timeseries ID on a given protocol arm. 
Ideally we should be able to parse the above message and extract the relevant information but for now you will have 
to input these manually yourself.

In the above example, the message is "constituent DefaultComp.CO_2 on arm baz is missing from the scoring evaluation context"
so the protocol arm ID is "baz" and the timeseries ID is "DefaultComp.CO_2"

In [ ]:
# protocol arm ID of interest
arm_id = "baz"
# timeseries ID of interest
timeseries_id = "DefaultComp.CO_2"

model_results = calibration.results.timeseries_per_patient(
    iteration=iteration,
    patient_id=patient_id,
    select=[timeseries_id],
)
model_result = next(
    (item for item in model_results if item["scenarioArm"] == arm_id),
    None,
)
if model_result is None:
    print(f"No ModelResult found for arm {arm_id}")
else:
    print(model_result["error"])
